# Fraud Detection — LightGBM Training on Feast Offline Features

Trains a LightGBM binary classifier on **point-in-time features from the Feast offline
store**, joined to ground-truth labels in `application.labels`.

**Pipeline**
1. **Build the entity spine** — one row per *labeled* transaction (entity join keys,
   `event_timestamp`, `label`).
2. **Load features** — Feast `get_historical_features` (point-in-time join) or, while that
   is too slow over the full history, a direct-Postgres fallback (toggle `USE_FEAST`).
3. **Version the dataset** — snapshot the training frame to `dataset/` and push it with DVC.
4. **Encode** categoricals and select feature columns.
5. **Split** temporally (train on earlier rows, validate on the most recent).
6. **Train** a LightGBM classifier (binary: fraud vs. legit), early-stopping on PR-AUC.
7. **Evaluate** on the held-out validation slice.
8. **Save & register** — `models/model.bst` + schema, and log/register the run to MLflow.

**Design notes**
- Raw entity IDs are used only to build the spine and are dropped from the feature matrix.
- `is_declined` is excluded — post-authorization status is leakage for a pre-auth scorer.


In [ ]:
import json
import os
import subprocess
import time
import warnings
import yaml
from pathlib import Path

import lightgbm as lgb
import mlflow
import mlflow.lightgbm
import numpy as np
import pandas as pd
import psycopg
from feast import FeatureStore
from mlflow.models import infer_signature
from sklearn.metrics import average_precision_score, roc_auc_score

warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
CURRENT_DIR = Path().cwd()
REPO_ROOT = CURRENT_DIR.parents[1] if CURRENT_DIR.name == "training" else CURRENT_DIR
MODEL_DIR = REPO_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
DATASET_DIR = REPO_ROOT / "dataset"
DATASET_DIR.mkdir(parents=True, exist_ok=True)
FEAST_REPO = REPO_ROOT / "src" / "feature_store"


def load_env_file(path: Path) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


load_env_file(REPO_ROOT / ".env")

CONNINFO = (
    f"host={os.getenv('POSTGRES_HOST')} port={os.getenv('POSTGRES_PORT')} "
    f"user={os.getenv('POSTGRES_USER')} password={os.getenv('POSTGRES_PASSWORD')} "
    f"dbname={os.getenv('POSTGRES_DB')}"
)

print(f"Connecting to Postgres with: {CONNINFO}")


def log(msg: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000")
MLFLOW_EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME", "fraud-detection")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

log(f"Feast repo    -> {FEAST_REPO}")
log(f"Model output  -> {MODEL_DIR}")
log(f"Dataset dir   -> {DATASET_DIR}")
log(f"MLflow server -> {MLFLOW_TRACKING_URI} (experiment: {MLFLOW_EXPERIMENT_NAME})")

In [ ]:
SEED = 42
TARGET = "label"
FEATURE_SERVICE = "fraud_detection_service"
MODEL_NAME = os.getenv("MODEL_NAME", "fraud-detection")

USE_FEAST = False

NON_FEATURE_COLS = {
    "transaction_id", "user_id", "card_id", "merchant_id", "device_id",
    "event_timestamp", "computed_at", "is_declined", "account_created_at", "card_created_at", "user_country", TARGET,
}

CATEGORICAL_COLS = [
    "channel", "card_brand", "card_type",
    "customer_segment", "merchant_category",
    "is_virtual", "email_verified",
]

LGB_PARAMS = {
    "objective": "binary",
    "metric": ["average_precision", "auc"],
    "boosting_type": "gbdt",
    "learning_rate": 0.02,
    "num_leaves": 128,
    "min_child_samples": 100,
    "feature_fraction": 0.7,
    "bagging_fraction": 0.85,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 0.1,
    "verbose": -1,
    "n_jobs": -1,
    "seed": SEED,
}
NUM_BOOST_ROUND = 5000
EARLY_STOPPING = 200
VAL_FRAC = 0.20


## 1 — Build the entity spine (labeled transactions)

The spine is the *entity dataframe* Feast joins features onto: one row per labeled
transaction with the entity join keys, the point-in-time `event_timestamp` (`created_at`),
and the ground-truth `label`. It is bounded to a short, fully-labeled time window
(`SPINE_START` .. `SPINE_END`) — Feast's Postgres point-in-time join is expensive over the
full history, so a recent window keeps retrieval fast. (Only used when `USE_FEAST` is on.)

In [4]:
SPINE_START = "2026-06-27 00:00:00+00"
SPINE_END = "2026-06-29 00:00:00+00"

SPINE_SQL = """
    SELECT f.transaction_id,
           f.user_id,
           f.card_id,
           f.created_at AS event_timestamp,
           l.label
    FROM application.transaction_features f
    JOIN application.labels l ON l.transaction_id = f.transaction_id
    WHERE l.label IS NOT NULL
      AND f.created_at >= %(start)s
      AND f.created_at <  %(end)s
    ORDER BY f.created_at
"""


def load_spine() -> pd.DataFrame:
    log(f"Building entity spine (window {SPINE_START[:10]} .. {SPINE_END[:10]}) ...")
    with psycopg.connect(CONNINFO) as conn:
        spine = pd.read_sql(SPINE_SQL, conn, params={"start": SPINE_START, "end": SPINE_END})
    spine["event_timestamp"] = pd.to_datetime(spine["event_timestamp"], utc=True)
    spine[TARGET] = spine[TARGET].astype(np.int8)
    pos = int(spine[TARGET].sum())
    log(f"  spine rows: {len(spine):,}  positives: {pos:,} ({pos / len(spine):.3%})")
    return spine.reset_index(drop=True)


spine = load_spine()
spine.head()


[09:06:05] Building entity spine (window 2026-06-27 .. 2026-06-29) ...


KeyboardInterrupt: 

## 2 — Load the labeled feature frame

With `USE_FEAST = True`, `get_historical_features` runs a point-in-time join against the
offline store, returning each transaction's features as of its own `created_at`, and the
labels are merged back by `transaction_id`.

> ⚠️ **Currently `USE_FEAST = False`** — the Postgres offline point-in-time join is too slow
> over the full history, so we load all labeled features directly from Postgres (same
> columns, whole history). Flip `USE_FEAST` to `True` to restore the Feast path.

In [ ]:
FALLBACK_SQL = """
    SELECT f.*, l.label
    FROM application.transaction_features f
    JOIN application.labels l ON l.transaction_id = f.transaction_id
    WHERE l.label IS NOT NULL
    ORDER BY f.created_at
"""


def load_training_frame(spine: pd.DataFrame) -> pd.DataFrame:
    if USE_FEAST:
        log(f"Feast point-in-time join via feature service '{FEATURE_SERVICE}' ...")
        store = FeatureStore(repo_path=str(FEAST_REPO))
        service = store.get_feature_service(FEATURE_SERVICE)
        entity_df = spine[["transaction_id", "user_id", "card_id", "event_timestamp"]]
        features = store.get_historical_features(entity_df=entity_df, features=service).to_df()
        df = features.merge(spine[["transaction_id", TARGET]], on="transaction_id", how="inner")
    else:
        log("Loading all labeled features directly from Postgres ...")
        with psycopg.connect(CONNINFO) as conn:
            df = pd.read_sql(FALLBACK_SQL, conn)
        df = df.rename(columns={"created_at": "event_timestamp"})

    df["event_timestamp"] = pd.to_datetime(df["event_timestamp"], utc=True)
    df[TARGET] = df[TARGET].astype(np.int8)
    df = (
        df.drop_duplicates(subset="transaction_id")
          .sort_values("event_timestamp")
          .reset_index(drop=True)
    )
    log(f"  loaded {len(df):,} rows x {df.shape[1]} cols")
    return df


df = load_training_frame(spine)
df.head()


[09:00:39] Loading all labeled features directly from Postgres ...


## 3 — Version the training dataset with DVC

Snapshot the exact frame this run trains on to `dataset/training_data.parquet`, then hand
it to DVC (`dvc add` + `dvc push`). DVC keeps the large data out of git (tracking only the
small `.dvc` pointer) and versions each run's snapshot in the remote.

> Requires a DVC remote (`dvc remote add -d <name> <url>`). Without one, `dvc push` is
> reported as failed and skipped — the local snapshot and `.dvc` pointer are still created.

In [ ]:
def run_dvc(*args) -> int:
    try:
        result = subprocess.run(
            ["dvc", *args], cwd=str(REPO_ROOT), capture_output=True, text=True
        )
    except FileNotFoundError:
        log("dvc not found on PATH - skipping: " + " ".join(args))
        return -1
    label = "dvc " + " ".join(a for a in args if not a.startswith("-") or len(a) < 20)
    if result.returncode == 0:
        log(f"{label}: ok")
    else:
        lines = (result.stderr or result.stdout).strip().splitlines()
        log(f"{label}: FAILED - {lines[-1] if lines else 'unknown error'}")
    return result.returncode


dataset_path = DATASET_DIR / "training_data.parquet"
df.to_parquet(dataset_path, index=False)
log(f"Wrote {len(df):,} rows -> {dataset_path}")

dvc_pointer = DATASET_DIR / "training_data.parquet.dvc"
run_dvc("add", "dataset/training_data.parquet")
if run_dvc("push") != 0:
    log("Configure a DVC remote to enable push: dvc remote add -d <name> <url>")
    
dataset_md5 = None
if dvc_pointer.exists():
    meta = yaml.safe_load(dvc_pointer.read_text())
    dataset_md5 = meta["outs"][0]["md5"]
    log(f"dataset version (md5): {dataset_md5}")

## 4 — Encode categoricals and select feature columns

In [7]:
def encode_categoricals(df: pd.DataFrame):
    encoders = {}
    cat_cols = []
    for col in CATEGORICAL_COLS:
        if col not in df.columns:
            continue
        s = df[col].astype("string").fillna("missing").astype(str)
        mapping = {v: i for i, v in enumerate(sorted(s.unique()))}
        df[col] = s.map(mapping).astype(np.int32)
        encoders[col] = mapping
        cat_cols.append(col)
    log(f"Label-encoded {len(cat_cols)} categorical columns.")
    return cat_cols, encoders


def feature_columns(df: pd.DataFrame):
    return [c for c in df.columns if c not in NON_FEATURE_COLS]


cat_cols, encoders = encode_categoricals(df)
feat_cols = feature_columns(df)
log(f"{len(feat_cols)} feature columns ({len(cat_cols)} categorical)")
feat_cols


[09:07:27] Label-encoded 7 categorical columns.
[09:07:27] 29 feature columns (7 categorical)


['amount_usd',
 'log_amount',
 'hour',
 'weekday',
 'is_night',
 'channel',
 'card_brand',
 'card_type',
 'is_virtual',
 'customer_segment',
 'kyc_level',
 'email_verified',
 'merchant_category',
 'merchant_risk_level',
 'account_age_days',
 'card_age_days',
 'geo_mismatch',
 'foreign_ip',
 'recipient_differs',
 'card_tx_count_1h',
 'card_tx_count_24h',
 'card_amount_sum_24h',
 'card_seconds_since_last_tx',
 'card_amount_zscore',
 'card_tx_seq',
 'card_declines_24h',
 'user_tx_count_24h',
 'user_amount_sum_24h',
 'user_seconds_since_last_tx']

## 5 — Temporal train / validation split

A chronological split (train on the earliest `1 - VAL_FRAC`, validate on the most recent
`VAL_FRAC`) mirrors production, where the model only ever scores transactions that occur
after the ones it was trained on. A random split would leak future behaviour.

In [8]:
def temporal_split(df: pd.DataFrame, val_frac: float = VAL_FRAC):
    n = len(df)
    cut = int(n * (1.0 - val_frac))
    idx = np.arange(n)
    tr, va = idx[:cut], idx[cut:]
    log(f"Split - train: {len(tr):,}  val: {len(va):,} (most-recent {val_frac:.0%})")
    return tr, va


def precision_at_k(y_true: np.ndarray, y_score: np.ndarray, k_frac: float) -> float:
    k = max(1, int(len(y_score) * k_frac))
    top = np.argpartition(-y_score, k - 1)[:k]
    return float(y_true[top].sum()) / k


train_idx, val_idx = temporal_split(df)


[09:07:30] Split - train: 190,055  val: 47,514 (most-recent 20%)


## 6 — Train on the training set, early-stop on validation

Fraud is heavily imbalanced (~0.5% positives), so `scale_pos_weight` is set from the
training split's class ratio. Early stopping tracks **PR-AUC** (`average_precision`):
ROC-AUC saturates within a few trees under the heavy class weight and would stop training
far too early. The run is opened here and stays active across the eval + save cells.

In [ ]:
X, y = df[feat_cols], df[TARGET].to_numpy()
cat_idx = [c for c in cat_cols if c in feat_cols]

pos = int(y[train_idx].sum())
scale_pos_weight = (len(train_idx) - pos) / max(1, pos)
params = {**LGB_PARAMS, "scale_pos_weight": scale_pos_weight}
log(f"scale_pos_weight = {scale_pos_weight:.1f}  (train positives: {pos:,})")

mlflow.end_run()
run = mlflow.start_run(run_name=f"lgbm-{time.strftime('%Y%m%d-%H%M%S')}")
model_settings = {
    "name": os.getenv("MODEL_NAME", "fraud-detection"),
    "implementation": "mlserver_lightgbm.LightGBMModel",
    "parameters": {
        "uri": "./model.bst",
        "version": run.info.run_id[:8],   # version = mlflow run id
    },
}
dataset = mlflow.data.from_pandas(
    df,
    source="gs://fraud-detection-dvc/dataset",  # DVC remote (.dvc/config)
    name="fraud_training_data",
    targets=TARGET,               # cột nhãn -> UI biết đâu là target
    digest=dataset_md5,           # <- dùng md5 của DVC làm digest (khớp GCS)
)
mlflow.log_input(dataset, context="training")
mlflow.set_tags({
    "model_type": "lightgbm",
    "task": "binary_fraud_classification",
    "data_source": "feast" if USE_FEAST else "postgres_direct",
})
mlflow.log_params({
    **params,
    "num_boost_round": NUM_BOOST_ROUND,
    "early_stopping_rounds": EARLY_STOPPING,
    "val_frac": VAL_FRAC,
    "n_features": len(feat_cols),
    "n_train": int(len(train_idx)),
    "n_val": int(len(val_idx)),
})
log(f"MLflow run started: {run.info.run_id}")

dtrain = lgb.Dataset(X.iloc[train_idx], label=y[train_idx], categorical_feature=cat_idx)
dvalid = lgb.Dataset(X.iloc[val_idx], label=y[val_idx], categorical_feature=cat_idx,
                     reference=dtrain)

booster = lgb.train(
    params,
    dtrain,
    num_boost_round=NUM_BOOST_ROUND,
    valid_sets=[dtrain, dvalid],
    valid_names=["train", "valid"],
    callbacks=[
        lgb.early_stopping(EARLY_STOPPING, first_metric_only=True),
        lgb.log_evaluation(period=100),
    ],
)
log(f"Best iteration: {booster.best_iteration}")

## 7 — Evaluate on the held-out validation set

In [ ]:
val_pred = booster.predict(X.iloc[val_idx], num_iteration=booster.best_iteration)
y_val = y[val_idx]

metrics = {
    "val_roc_auc": float(roc_auc_score(y_val, val_pred)),
    "val_pr_auc": float(average_precision_score(y_val, val_pred)),
    "val_precision_at_1pct": precision_at_k(y_val, val_pred, 0.01),
    "val_precision_at_0.5pct": precision_at_k(y_val, val_pred, 0.005),
    "best_iteration": int(booster.best_iteration),
    "n_features": len(feat_cols),
    "n_train": int(len(train_idx)),
    "n_val": int(len(val_idx)),
    "val_positives": int(y_val.sum()),
    "scale_pos_weight": float(scale_pos_weight),
}
mlflow.log_metrics({k: float(v) for k, v in metrics.items()})
print(json.dumps(metrics, indent=2))

## 8 — Save the model, its schema, and register in MLflow

The evaluated booster is written to `models/model.bst` alongside the metrics and a feature
schema (column order + categorical encoders) so serving can rebuild an identical vector.
The booster, params, metrics and artifacts are logged to MLflow, the model is **registered**
under `MODEL_NAME` (a new version every run), and the run is closed.

In [ ]:
booster.save_model(str(MODEL_DIR / "model.bst"), num_iteration=booster.best_iteration)

schema = {
    "version": 1,
    "offline_source": "feast::fraud_detection_service",
    "feature_view": "transaction_features",
    "label_table": "application.labels",
    "target": TARGET,
    "feature_columns": feat_cols,
    "categorical_features": cat_cols,
    "categorical_encoders": encoders,
    "lgb_params": LGB_PARAMS,
}

(MODEL_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
(MODEL_DIR / "feature_schema.json").write_text(json.dumps(schema, indent=2))
(MODEL_DIR / "model-settings.json").write_text(json.dumps(model_settings, indent=2))
log(f"Saved model.bst + metrics.json + feature_schema.json + model-settings.json to {MODEL_DIR}")

sample = X.iloc[val_idx].head(200)
signature = infer_signature(sample, booster.predict(sample))
model_info = mlflow.lightgbm.log_model(
    booster,
    artifact_path="model",
    signature=signature,
    input_example=X.iloc[val_idx].head(5),
    registered_model_name=MODEL_NAME,
)
for name in ("model.bst", "metrics.json", "feature_schema.json", "model-settings.json"):
    mlflow.log_artifact(str(MODEL_DIR / name))

run_id = mlflow.active_run().info.run_id
mlflow.end_run()
log(f"Logged + registered model '{MODEL_NAME}' from run {run_id} ({MLFLOW_TRACKING_URI})")
if getattr(model_info, "registered_model_version", None):
    log(f"Registered version: {MODEL_NAME} v{model_info.registered_model_version}")